# Investor Questions Analysis

Analysis of rental property as an investment. All queries build gold tables in `wanderbricks_training.bookings` from source data in `samples.wanderbricks`.

| Table | Description |
|---|---|
| `gold_host_performance` | Host activity: properties, bookings, revenue and ratings per host (2024 onwards) |
| `gold_bookings_by_year` | Booking counts per year — used to determine which years we need property price data for |
| `gold_revenue_and_pricing_by_country` | Revenue per property, average nightly rate and booking counts by country and year |
| `gold_rental_duration_and_trends` | Average rental duration and booking growth/decline by country and year |
| `gold_seasonal_demand` | Booking counts by season, per country and overall |
| `gold_real_estate_market` | House price index (OECD, 13 countries) and price per m² (Numbeo, 18 countries) |

## 1. Host Performance
Assignment 2 — host activity analysis: properties, bookings, revenue and ratings per host, for 2024 onwards.

In [0]:
%sql
DROP TABLE IF EXISTS wanderbricks_training.bookings.gold_host_performance;

CREATE TABLE wanderbricks_training.bookings.gold_host_performance
COMMENT 'Top-performing hosts with properties, bookings, revenue and ratings (2024 onwards). Base table for Assignment 2.'
AS
-- Analyze top-performing hosts with their properties and bookings
SELECT
  h.host_id,
  h.name as host_name,
  h.country as host_country,
  h.rating as host_rating,
  h.is_verified,
  COUNT(DISTINCT p.property_id) as properties_count,
  COUNT(DISTINCT b.booking_id) as total_bookings,
  ROUND(SUM(b.total_amount), 2) as total_revenue,
  ROUND(AVG(r.rating), 2) as avg_property_rating,
  COUNT(DISTINCT r.review_id) as review_count
FROM samples.wanderbricks.hosts h
INNER JOIN samples.wanderbricks.properties p ON h.host_id = p.host_id
LEFT JOIN samples.wanderbricks.bookings b ON p.property_id = b.property_id
  AND b.status = 'confirmed'
LEFT JOIN samples.wanderbricks.reviews r ON p.property_id = r.property_id
  AND r.created_at >= '2024-01-01'
  AND r.is_deleted = false
WHERE h.is_active = true
GROUP BY h.host_id, h.name, h.country, h.rating, h.is_verified
HAVING total_bookings > 0
ORDER BY total_revenue DESC;

## 2. Bookings by Year
Identifies which years have booking data, so we know which period the property prices need to cover.

In [0]:
%sql
DROP TABLE IF EXISTS wanderbricks_training.bookings.gold_bookings_by_year;

CREATE TABLE wanderbricks_training.bookings.gold_bookings_by_year
COMMENT 'Booking counts per year. Used to determine which years we need property price data for.'
AS
SELECT
  YEAR(check_in) as booking_year,
  COUNT(DISTINCT booking_id) as total_bookings,
  MIN(check_in) as earliest_check_in,
  MAX(check_in) as latest_check_in
FROM samples.wanderbricks.bookings
WHERE status = 'confirmed'
GROUP BY YEAR(check_in)
ORDER BY booking_year;

## 3. Revenue and Pricing by Country
Category 1 — Profitability and financial performance.
Revenue per property, average nightly rate and booking counts, broken down by country and year.

In [0]:
%sql
DROP TABLE IF EXISTS wanderbricks_training.bookings.gold_revenue_and_pricing_by_country;

CREATE TABLE wanderbricks_training.bookings.gold_revenue_and_pricing_by_country
COMMENT 'Rental revenue, average nightly rate and booking counts by country and year. Base table for profitability analysis (Category 1).'
AS
SELECT
  h.country,
  YEAR(b.check_in) as booking_year,
  COUNT(DISTINCT p.property_id) as properties_count,
  COUNT(DISTINCT b.booking_id) as total_bookings,
  ROUND(SUM(b.total_amount), 2) as total_revenue,
  ROUND(SUM(b.total_amount) / COUNT(DISTINCT p.property_id), 2) as avg_revenue_per_property,
  ROUND(AVG(p.base_price), 2) as avg_price_per_night
FROM samples.wanderbricks.hosts h
INNER JOIN samples.wanderbricks.properties p ON h.host_id = p.host_id
INNER JOIN samples.wanderbricks.bookings b ON p.property_id = b.property_id
WHERE b.status = 'confirmed'
GROUP BY h.country, YEAR(b.check_in)
ORDER BY h.country, booking_year;

## 4. Rental Duration and Trends
Category 2 — Tenant behavior and retention.
Average rental duration in nights, plus year-over-year growth or decline in bookings per country.

In [0]:
%sql
DROP TABLE IF EXISTS wanderbricks_training.bookings.gold_rental_duration_and_trends;

CREATE TABLE wanderbricks_training.bookings.gold_rental_duration_and_trends
COMMENT 'Average rental duration and booking counts by country and year, including year-over-year growth. Base table for tenant behavior analysis (Category 2).'
AS
WITH yearly_stats AS (
  SELECT
    h.country,
    YEAR(b.check_in) as booking_year,
    COUNT(DISTINCT b.booking_id) as total_bookings,
    ROUND(AVG(DATEDIFF(b.check_out, b.check_in)), 2) as avg_rental_duration_nights
  FROM samples.wanderbricks.hosts h
  INNER JOIN samples.wanderbricks.properties p ON h.host_id = p.host_id
  INNER JOIN samples.wanderbricks.bookings b ON p.property_id = b.property_id
  WHERE b.status = 'confirmed'
  GROUP BY h.country, YEAR(b.check_in)
)

SELECT
  country,
  booking_year,
  avg_rental_duration_nights,
  total_bookings,
  LAG(total_bookings) OVER (PARTITION BY country ORDER BY booking_year) as prev_year_bookings,
  ROUND(
    (total_bookings - LAG(total_bookings) OVER (PARTITION BY country ORDER BY booking_year))
    / LAG(total_bookings) OVER (PARTITION BY country ORDER BY booking_year) * 100,
    2
  ) as bookings_growth_pct
FROM yearly_stats
ORDER BY country, booking_year;

## 5. Seasonal Demand
Category 3 — Demand and seasonality.
Booking counts by season, both per country and as a share of each country's total bookings.

In [0]:
%sql
DROP TABLE IF EXISTS wanderbricks_training.bookings.gold_seasonal_demand;

CREATE TABLE wanderbricks_training.bookings.gold_seasonal_demand
COMMENT 'Booking counts by season, per country and across all countries. Base table for seasonality analysis (Category 3).'
AS
WITH seasonal_bookings AS (
  SELECT
    h.country,
    CASE
      WHEN MONTH(b.check_in) IN (12, 1, 2) THEN 'Winter'
      WHEN MONTH(b.check_in) IN (3, 4, 5) THEN 'Spring'
      WHEN MONTH(b.check_in) IN (6, 7, 8) THEN 'Summer'
      WHEN MONTH(b.check_in) IN (9, 10, 11) THEN 'Autumn'
    END as season,
    b.booking_id,
    b.total_amount
  FROM samples.wanderbricks.hosts h
  INNER JOIN samples.wanderbricks.properties p ON h.host_id = p.host_id
  INNER JOIN samples.wanderbricks.bookings b ON p.property_id = b.property_id
  WHERE b.status = 'confirmed'
)

SELECT
  country,
  season,
  COUNT(DISTINCT booking_id) as total_bookings,
  ROUND(SUM(total_amount), 2) as total_revenue,
  ROUND(
    COUNT(DISTINCT booking_id) * 100.0
    / SUM(COUNT(DISTINCT booking_id)) OVER (PARTITION BY country),
    2
  ) as pct_of_country_bookings
FROM seasonal_bookings
GROUP BY country, season
ORDER BY country, total_bookings DESC;

## 6. Real Estate Market
Category 4 — Property market.
Combines the OECD house price index (13 countries, 2023–2025) with current price per m² from Numbeo (18 countries).

Note: this cell is Python, not SQL — the notebook must run on a cluster, not a SQL Warehouse.

In [0]:
import requests
import pandas as pd
from io import StringIO

url = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.ECO.MPD,DSD_AN_HOUSE_PRICES@DF_HOUSE_PRICES,1.0/all"
    "?startPeriod=2023&endPeriod=2025"
    "&dimensionAtObservation=AllDimensions"
    "&format=csvfilewithlabels"
)

response = requests.get(url)
response.raise_for_status()
oecd_df = pd.read_csv(StringIO(response.text))

oecd_countries = [
    "United States", "Japan", "Germany", "France", "Spain",
    "Italy", "United Kingdom", "Australia", "India", "Greece",
    "Canada", "Austria", "Switzerland"
]
target_years = ["2023", "2024", "2025"]

oecd_df = oecd_df[
    (oecd_df["Reference area"].isin(oecd_countries)) &
    (oecd_df["TIME_PERIOD"].astype(str).str[:4].isin(target_years))
]

oecd_yearly = (
    oecd_df
    .assign(year=oecd_df["TIME_PERIOD"].astype(str).str[:4])
    .groupby(["Reference area", "year"], as_index=False)["OBS_VALUE"]
    .mean()
    .rename(columns={"Reference area": "country", "OBS_VALUE": "price_index"})
)
oecd_yearly["price_index"] = oecd_yearly["price_index"].round(2)

numbeo_data = {
    "country": [
        "United States", "Japan", "Thailand", "Germany", "France", "Spain",
        "Italy", "Egypt", "China", "United Kingdom", "Australia", "India",
        "United Arab Emirates", "Greece", "Singapore", "Canada",
        "Austria", "Switzerland"
    ],
    "price_per_sqm_eur": [
        3311.76, 5438.55, 3531.87, 6321.46, 5688.14, 4324.04,
        3788.38, 756.37, 4740.89, 5766.04, 8394.43, 1231.85,
        5814.89, 3191.17, 23343.76, 4746.80,
        7585.88, 19805.46
    ]
}
numbeo_df = pd.DataFrame(numbeo_data)

market_df = numbeo_df.merge(oecd_yearly, on="country", how="left")

baseline = (
    oecd_yearly[oecd_yearly["year"] == "2023"]
    [["country", "price_index"]]
    .rename(columns={"price_index": "index_2023"})
)
market_df = market_df.merge(baseline, on="country", how="left")
market_df["index_change_pct"] = (
    (market_df["price_index"] - market_df["index_2023"]) / market_df["index_2023"] * 100
).round(2)

spark_df = spark.createDataFrame(market_df)
spark_df.write.mode("overwrite").saveAsTable(
    "wanderbricks_training.bookings.gold_real_estate_market"
)
spark_df.display()